In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ClinicalDistill — Phi-2 LoRA
## Experiment 4

### Why Phi-2?
We've run Gemma-3-1B with LoRA and QLoRA. Now we swap the model family.

**Phi-2** (microsoft/phi-2):
- 2.7B parameters — ~2.7x larger than Gemma-3-1B
- Trained on synthetic textbook-quality data ("textbooks are all you need")
- Known to punch above its weight on reasoning tasks
- Different architecture family from Gemma — gives us a genuine cross-family comparison

### What we're testing
Same dataset. Same LoRA config. Same eval.
The only variable is the model.

This gives us one more row in the paper's results table:
| Model | Params | F1 | Urgent Acc |
|---|---|---|---|
| Gemma-3-1B LoRA | 1B | 0.781 | 85.7% |
| Gemma-3-1B QLoRA | 1B | 0.740 | 82.9% |
| Phi-2 LoRA | 2.7B | ___ | ___ |

In [2]:
import os

PROJECT_PATH = '/content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill'

print(os.listdir(PROJECT_PATH))
!wc -l {PROJECT_PATH}/data/train_fixed.jsonl {PROJECT_PATH}/data/test_fixed.jsonl

['README.md', 'requirements.txt', '.env', '.gitignore', 'data', 'schema', '.git', 'models']
  145 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/train_fixed.jsonl
   35 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/test_fixed.jsonl
  180 total


In [3]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.5 MB/s eta 0:00:00


In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

True
Tesla T4
VRAM: 15.6 GB


In [5]:
from datasets import load_dataset
import json

# Same canonical dataset — do not change
dataset = load_dataset("json", data_files={
    "train": f"{PROJECT_PATH}/data/train_fixed.jsonl",
    "test":  f"{PROJECT_PATH}/data/test_fixed.jsonl"
})

print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 145
    })
    test: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 35
    })
})
{'input': 'Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.', 'output': {'symptoms': ['chest pain', 'shortness of breath'], 'duration': ['2 hours', 'unspecified'], 'severity': ['severe', 'unspecified'], 'urgent': True}, 'metadata': {'clinical_domain': 'cardiac', 'generated_by': 'gpt-4o'}}


In [6]:
def format_example(example):
    output = example['output']
    if isinstance(output, str):
        output = json.loads(output)

    # Identical prompt format to all previous experiments
    return {
        "text": f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{example['input']}</input>
<output>{json.dumps(output)}</output>"""
    }

dataset = dataset.map(format_example)
print(dataset["train"][0]["text"])

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.</input>
<output>{"symptoms": ["chest pain", "shortness of breath"], "duration": ["2 hours", "unspecified"], "severity": ["severe", "unspecified"], "urgent": true}</output>


In [7]:
from huggingface_hub import login
import json
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load Phi-2

Phi-2 uses a different tokenizer than Gemma — it doesn't have a pad token by default.
We set `pad_token = eos_token` which is standard practice for causal LMs without a pad token.

Phi-2 also uses eager attention (no flash attention) — set `attn_implementation='eager'` to avoid warnings.

In [21]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import torch

model_id = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Add a dedicated pad token instead of reusing eos_token
# When pad_token = eos_token, the data collator masks EOS tokens as padding
# which cascades to masking the entire sequence → loss = 0
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
tokenizer.padding_side = "right"

# Load config first, set pad_token_id on it — Phi-2 doesn't accept it as a kwarg
config = AutoConfig.from_pretrained(model_id)
config.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager"
)

# Resize embeddings to account for the new [PAD] token
model.resize_token_embeddings(len(tokenizer))

print(f"Model loaded: {model_id}")
print(f"Parameters: {model.num_parameters():,}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Model loaded: microsoft/phi-2
Parameters: 2,775,054,456
VRAM used: 11.20 GB


In [10]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## LoRA Configuration

Phi-2's attention layer names are different from Gemma's.
- Gemma uses: `q_proj`, `v_proj`
- Phi-2 uses: `q_proj`, `v_proj` inside `MixedMLP` blocks — same names, same approach

Keeping r=16, alpha=32 identical to Gemma experiments for fair comparison.

In [22]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Phi-2 is 2.7B params — expect higher trainable param count than Gemma-1B

trainable params: 5,242,880 || all params: 2,780,297,336 || trainable%: 0.1886


In [19]:
from trl import SFTTrainer, SFTConfig
import time

OUTPUT_DIR = f"{PROJECT_PATH}/models/phi2-lora"

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
    # max_seq_length=512
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

start_time = time.time()
print("Starting Phi-2 LoRA training...")
trainer.train()
training_time = time.time() - start_time

print(f"Training complete!")
print(f"Training time: {training_time:.0f}s ({training_time/60:.1f} min)")

Adding EOS to train dataset:   0%|          | 0/145 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/145 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting Phi-2 LoRA training...


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000


Training complete!
Training time: 285s (4.8 min)


In [20]:
SAVE_PATH = f"{PROJECT_PATH}/models/phi2-lora-final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Phi-2 LoRA model saved to: {SAVE_PATH}")

Phi-2 LoRA model saved to: /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/models/phi2-lora-final
